In [2]:
### What is MCP
# defined by anthropic, defining a standard for how LLM applications connect to your tools and data sources
# the standard is built on a client/server relationship - in our case, the MCP client is the Agent, and the MCP server are the Tools, Resources, and Prompts that the MCP server exposes to the client
# Once we've built an MCP server, it's very easy to share it with other clients, e.g. Agents or ChatBots.
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

# See the following 2.1 notebook for details on stdio vs http_stream transports
client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [6]:
# The client still uses await (async) when defining tools, resources, and prompt. This is because the client library is async regardless of how the server functions are written.
# get_tools(), get_resource(), and get_prompt() are coroutines - the "await" means “start this coroutine, and if it has to wait on something (network, disk, a timer), pause me here and let the event loop run something else. Resume me when the result is ready.”
# If you forget the "await", you will just get the coroutine object back rather than the result, e.g. the below for get_tools() without "await".
# LangChain will also throw an error "/var/folders/1s/j40qd47n2759r8sm0jytjy6w0000gp/T/ipykernel_60694/2569037413.py:6: RuntimeWarning: coroutine 'MultiServerMCPClient.get_tools' was never awaited
#  tools = client.get_tools()" 
# <coroutine object MultiServerMCPClient.get_tools at 0x1123ce6c0>
# Server functions (see resources/2.1_mcp_server.py), can be written sync or async, depending on the kind of task the server is performing - e.g. it would make sense to use async functions if the tool does I/O operationlike HTTP calls or DB queries

# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

print(tools)
print(resources)
print(prompt)

<coroutine object MultiServerMCPClient.get_tools at 0x1123ce6c0>
<coroutine object MultiServerMCPClient.get_resources at 0x11254fc40>
<coroutine object MultiServerMCPClient.get_prompt at 0x112536540>


/var/folders/1s/j40qd47n2759r8sm0jytjy6w0000gp/T/ipykernel_60694/2569037413.py:6: RuntimeWarning: coroutine 'MultiServerMCPClient.get_tools' was never awaited
  tools = client.get_tools()
/var/folders/1s/j40qd47n2759r8sm0jytjy6w0000gp/T/ipykernel_60694/2569037413.py:9: RuntimeWarning: coroutine 'MultiServerMCPClient.get_resources' was never awaited
  resources = client.get_resources("local_server")
/var/folders/1s/j40qd47n2759r8sm0jytjy6w0000gp/T/ipykernel_60694/2569037413.py:12: RuntimeWarning: coroutine 'MultiServerMCPClient.get_prompt' was never awaited
  prompt = client.get_prompt("local_server", "prompt")


In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [ ]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [ ]:
from pprint import pprint

pprint(response)

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [ ]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [ ]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)